# Notes about Simons Algorithm 

- first register must be exactly like it was initially. 
- copying of first registers bit value can be copied to 2nd register bit by using cnot gates that will overwrite |0> into | f(x) > 
- say cnot gate between 1st qubit of first register and first qubit of 2nd register. if first qubit is 0.. 2nd qubit will also be zero. if first qubit is one the second will have X applied to the input and changed to 1. 
- running this circuit multiple times gives output for each time as z 
- we get systems of equation s.z = 0 s1.z1 = 0 s2.z2=0 and so on 
- we then solve that systems of equations classically by checking for orthogonality and then infer our secret string from that operation. z is not the secret string. 


In [213]:
from qiskit import QuantumCircuit, transpile 
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2
import numpy as np
from qiskit.visualization import plot_histogram


In [214]:
def stringDotProduct(str1, str2):
    result = 0
    for i in range(len(str1)):
        if (str1[i] == '1' and str2[i] == '1'):
            result += 1
    return (result%2)

def simonSolver(orthogonal_set):
    # This is basically the same as saying you have less equations than unknowns.
    if len(orthogonal_set) < len(orthogonal_set[0]):
        print("You need more strings!!!!!!!!")
        return ""

    # We define n to be the length of the strings. 
    # We assume all of them to be of the same length.
    n = len(orthogonal_set[0])

    # This basically goes over all possible integer values from 0 all the way
    # up to 2**n-1, the biggest number you can represent with n bits. For each
    # value, it converts it to binary form, padding the beginning with 0s to 
    # make its length n.
    setOfAllPossibleBitStrings = [ format(b, '0'+str(n)+'b') for b in range(2**n) ]

    # Using brute force method
    # the algorithm just goes over all possible bitstrings and checks
    # whether that particular string is orthogonal to each bitstring in the orthogonal_set.
    # If it is, it equates that to the result.
    # One important thing to notice is 000..0 bitstring is always orthonogal to any set
    # of bitstrings, and since setOfAllPossibleBitStrings contains the bitstrings in increasing
    # order, we return the latest one as our result.
    result = ""
    for string in setOfAllPossibleBitStrings:
        orthogonal_counter = 0
        for orthoElement in orthogonal_set:
            if (stringDotProduct(string, orthoElement) == 0):
                orthogonal_counter += 1

        if (orthogonal_counter == len(orthogonal_set)):
            result = string
    return result

# This function is used to generate the oracle for the Simon's algorithm.
def simonOracle(qc, n):
    # Generate a random bit string s with length n
    #s = "".join(['1' if np.random.randint(2) == 1 else '0' for _ in range(n)])
    s = "101"
    # Initialize the quantum circuit for the oracle
    oracle_circuit = QuantumCircuit(2*n)

    # Apply CNOT gates to copy the first register to the second register
    for i in range(n):
        oracle_circuit.cx(i, n+i)

    # Find indices where s[i] = 1
    setOfIndices = [i for i in range(n) if s[i] == "1"]

    # Check if s is not all zeros
    if setOfIndices:
        least_significant_index = setOfIndices[-1]

        # Apply CNOTs controlled by least_significant_index
        for target in setOfIndices:
            oracle_circuit.cx(least_significant_index, n+target)

        # Apply X gates to the second register
        for i in range(n):
            oracle_circuit.x(n+i)

    # Convert the oracle circuit to a gate with a label
    oracle_gate = oracle_circuit.to_gate()
    qc.append(oracle_gate, range(2*n))

    return s  # Return the secret string

def reverse_dict_keys(d):
    return {key[::-1]: value for key, value in d.items()}


# Make truth table of the oracle. we need this to confirm the secret string
# is fulfilling the promise. 

def truth_table(qc):
    truthTable= 0 
    return truthTable
    pass

def runCircuit(qc):
    backend = AerSimulator(method = 'automatic', precision = 'single')
    sampler = SamplerV2(mode=backend)
    qc = transpile(qc, backend=backend)
    job = sampler.run([qc])
    result = job.result()
    qc_counts = result[0].data.meas.get_counts()
    return qc_counts


In [215]:
# We need a total of 2n quantum bits, n for the input and n for the output.
n = 3
qc = QuantumCircuit(2*n)

for i in range(n): # Apply Hadamard gates to the first n qubits
    qc.h(i)
qc.barrier()

# Apply the oracle
secret_string = simonOracle(qc, n) 

qc.barrier()
qc.h(range(n))

# now measure all 
qc.measure_all()


qc_counts = runCircuit(qc)
print(reverse_dict_keys(qc_counts))
plot_histogram(qc_counts)

# The qc counts are not z strings.. z strings are the first registers values
# For each result we get.. least

orthogonal_set = []
for bitstring in qc_counts.keys():
    reversed_bitstring = bitstring[::-1]
    first_n_elements = reversed_bitstring[0:n]
    # we get only first the n qubit values like from 0011 we get 00. 
    # note that this is after reversing. so before it would 1100 
    # after reversing its 0011 and then we get 00 from it. that's equal to z string
    # We do not want repeated values so only getting unique values
    if not( first_n_elements in orthogonal_set):
        orthogonal_set.append( first_n_elements)

print(orthogonal_set)
# These generate a set that secret string s is orthogonal to. 


# all quantum part done. 
# run the result through simonsolver that return the s that is orthogonal to all elements 
# in the orthogonal_set list. 

secret_string_result = simonSolver(orthogonal_set)
print("Secret string we found is: ", secret_string_result)
print("Initial random secret string generated was: ", secret_string)

{'101111': 72, '111001': 48, '000101': 65, '101101': 54, '010001': 69, '010111': 77, '000011': 62, '111111': 66, '010101': 63, '101011': 58, '000111': 62, '101001': 69, '111011': 71, '111101': 68, '000001': 61, '010011': 59}
['101', '111', '000', '010']
Secret string we found is:  101
Initial random secret string generated was:  101


In [216]:
import qiskit 
print(qiskit.version.get_version_info())

1.3.2
